# Week 6 Practical Activity — Simple Information Extraction System
Extracts **PERSON**, **ORGANIZATION**, **LOCATION** and **DATE** entities from text using spaCy's pretrained English NER model.

## Student Tasks 1–3: Install Python, spaCy, and the English model
Run once (outside the notebook, in a terminal, or in a cell — shown here for completeness). The `en_core_web_sm` model is spaCy's small English pipeline and already includes a trained named-entity recognizer.

In [1]:
# Student Task 1-3 (run once):
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy

nlp = spacy.load("en_core_web_sm")

## Student Task 4: Create the Python program

In [2]:
CATEGORY_MAP = {
    "PERSON": "PERSON",
    "ORG": "ORGANIZATION",
    "GPE": "LOCATION",   # countries, cities, towns
    "LOC": "LOCATION",   # non-GPE locations (mountains, rivers, regions)
    "DATE": "DATE",
}


def extract_entities(text):
    """Return a list of (entity_text, category) for the four required
    categories only, in the order spaCy found them."""
    doc = nlp(text)
    results = []
    for ent in doc.ents:
        category = CATEGORY_MAP.get(ent.label_)
        if category:
            results.append((ent.text, category))
    return results


def print_table(results):
    width = max([len(t) for t, _ in results] + [len("Entity")]) + 2
    print(f"{'Entity':<{width}}| {'Category'}")
    print("-" * (width + 15))
    for text, category in results:
        print(f"{text:<{width}}| {category}")




## Student Task 5: Test the program using the supplied text

In [3]:
sample_text = (
    "Dr. Peter Mwangi attended the Artificial Intelligence Conference "
    "at Mount Kenya University in Thika on 20 September 2026."
)

print("Sample input:")
print(sample_text)
print()

results = extract_entities(sample_text)
print_table(results)

Sample input:
Dr. Peter Mwangi attended the Artificial Intelligence Conference at Mount Kenya University in Thika on 20 September 2026.

Entity                                  | Category
-------------------------------------------------------
Peter Mwangi                            | PERSON
the Artificial Intelligence Conference  | ORGANIZATION
Mount Kenya University                  | ORGANIZATION
20 September 2026                       | DATE


### Comparing against the expected output
| Entity | Expected Category | System Output |
|---|---|---|
| Peter Mwangi | PERSON | ✅ PERSON |
| Mount Kenya University | ORGANIZATION | ✅ ORGANIZATION |
| Thika | LOCATION | ❌ missed entirely |
| 20 September 2026 | DATE | ✅ DATE |

The system also pulled out *"the Artificial Intelligence Conference"* as ORGANIZATION — reasonable (it does name an organised body of people), though arguably it's closer to an **EVENT** (see the Extension section).

## Student Tasks 6–8: Five additional sentences, tested and recorded

In [4]:
sentences = [
    "President William Ruto opened the new stadium in Nairobi on "
    "15 March 2026.",
    "Amazon and Google are both investing in data centres across Kenya.",
    "Wangari Maathai founded the Green Belt Movement in 1977.",
    "The Nairobi Innovation Week will be held at the KICC on "
    "3 November 2026.",
    "Elon Musk visited Mombasa last January to discuss Starlink "
    "with Safaricom.",
]



In [5]:
for i, sentence in enumerate(sentences, start=1):
    print(f"\nSentence {i}: {sentence}")
    results = extract_entities(sentence)
    if results:
        print_table(results)
    else:
        print("  (no entities extracted)")


Sentence 1: President William Ruto opened the new stadium in Nairobi on 15 March 2026.
Entity         | Category
------------------------------
William Ruto   | PERSON
Nairobi        | LOCATION
15 March 2026  | DATE

Sentence 2: Amazon and Google are both investing in data centres across Kenya.
Entity  | Category
-----------------------
Amazon  | ORGANIZATION
Google  | ORGANIZATION
Kenya   | LOCATION

Sentence 3: Wangari Maathai founded the Green Belt Movement in 1977.
Entity                   | Category
----------------------------------------
Wangari Maathai          | PERSON
the Green Belt Movement  | ORGANIZATION
1977                     | DATE

Sentence 4: The Nairobi Innovation Week will be held at the KICC on 3 November 2026.
Entity                       | Category
--------------------------------------------
The Nairobi Innovation Week  | ORGANIZATION
3 November 2026              | DATE

Sentence 5: Elon Musk visited Mombasa last January to discuss Starlink with Safaricom.
Ent

## Student Task 9: Entities incorrectly identified or missed

| Sentence | Issue | Explanation |
|---|---|---|
| Sample sentence | **Thika** missed entirely | A less common Kenyan town; spaCy's small English model was trained mostly on US/UK news text (OntoNotes), so it under-recognises African place names it saw rarely or never in training. |
| 4 — Nairobi Innovation Week | Tagged **ORGANIZATION** instead of **EVENT** | The phrase *looks* like an institution name (capitalised multi-word noun phrase), which is exactly the surface pattern the model uses for ORG; it has very little signal to prefer EVENT without domain-specific training. |
| 4 — KICC | Missed entirely | An acronym with no expansion in the sentence; the model has little to no exposure to this specific Kenyan venue acronym. |
| 5 — Starlink | Missed entirely | A product/brand name; the base model doesn't reliably tag PRODUCT entities (see Extension). |

**Why this happens:** `en_core_web_sm` is a small, general-purpose model trained primarily on Western news and web text (the OntoNotes 5 corpus). It is good at well-represented patterns (full person names, well-known multinational companies, standard date formats) and weaker on: locally-specific place names, acronyms, and entities that need world knowledge the model never saw during training. A larger model (`en_core_web_trf`, spaCy's transformer-based pipeline) or fine-tuning on a Kenyan-news dataset would likely close some of these gaps.

---
## Extension Activity — MONEY, ORGANIZATION, PRODUCT, EVENT, TIME, PERCENTAGE
The same pipeline, extended to map a few more of spaCy's native NER labels.

In [6]:
EXTENDED_CATEGORY_MAP = {
    "PERSON": "PERSON",
    "ORG": "ORGANIZATION",
    "GPE": "LOCATION",
    "LOC": "LOCATION",
    "DATE": "DATE",
    "MONEY": "MONEY",
    "PRODUCT": "PRODUCT",
    "EVENT": "EVENT",
    "TIME": "TIME",
    "PERCENT": "PERCENTAGE",
}


def extract_entities_extended(text):
    doc = nlp(text)
    results = []
    for ent in doc.ents:
        category = EXTENDED_CATEGORY_MAP.get(ent.label_)
        if category:
            results.append((ent.text, category))
    return results


def print_table(results):
    if not results:
        print("  (no entities extracted)")
        return
    width = max([len(t) for t, _ in results] + [len("Entity")]) + 2
    print(f"{'Entity':<{width}}| {'Category'}")
    print("-" * (width + 15))
    for text, category in results:
        print(f"{text:<{width}}| {category}")


extension_sentences = [
    "Safaricom reported a profit of KSh 38 billion for the financial year, "
    "up 12 percent from last year.",
    "Apple launched the new iPhone 17 at 10:00 am during its September "
    "event.",
    "The Nairobi Innovation Week will host the Startup Pitch Event on "
    "3 November 2026.",
    "Tesla's Model 3 now costs about $35,000, a 5% drop in price.",
    "The World Economic Forum begins at 9:30 in the morning in Davos.",
]



In [7]:
for i, sentence in enumerate(extension_sentences, start=1):
    print(f"\nSentence {i}: {sentence}")
    print_table(extract_entities_extended(sentence))


Sentence 1: Safaricom reported a profit of KSh 38 billion for the financial year, up 12 percent from last year.
Entity              | Category
-----------------------------------
the financial year  | DATE
12 percent          | PERCENTAGE
last year           | DATE

Sentence 2: Apple launched the new iPhone 17 at 10:00 am during its September event.
Entity     | Category
--------------------------
Apple      | ORGANIZATION
10:00 am   | TIME
September  | DATE

Sentence 3: The Nairobi Innovation Week will host the Startup Pitch Event on 3 November 2026.
Entity                       | Category
--------------------------------------------
The Nairobi Innovation Week  | ORGANIZATION
3 November 2026              | DATE

Sentence 4: Tesla's Model 3 now costs about $35,000, a 5% drop in price.
Entity         | Category
------------------------------
Tesla          | ORGANIZATION
Model 3        | PRODUCT
about $35,000  | MONEY
5%             | PERCENTAGE

Sentence 5: The World Economic Forum b

### Extension — observations
- **MONEY**: `$35,000` was caught correctly, but `KSh 38 billion` was missed — the model recognises the `$` symbol pattern far more reliably than a Kenyan-shilling abbreviation, again reflecting its Western-centric training data.
- **PRODUCT**: `Model 3` was correctly tagged, but `iPhone 17` was not picked up at all — PRODUCT is one of the categories `en_core_web_sm` is explicitly weakest on, since product names are highly varied and fast-changing.
- **EVENT**: never once predicted across any test sentence, even for obvious cases like *"Startup Pitch Event"* — the small English model essentially defaults ambiguous multi-word capitalised phrases to ORG.
- **TIME** and **PERCENTAGE** worked reliably whenever the text used a conventional format (`10:00 am`, `12 percent`, `5%`).